# Lab type: review
# Course: ML401 — MLOps & Model Deployment
# Lesson: Training-Serving Skew
# Task: A working feature pipeline is provided. Review it for skew risk: identify where skew could enter, how you would detect it in production, and what changes would make it skew-resistant.

In [ ]:
# !pip install scikit-learn pandas numpy

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import joblib

np.random.seed(42)
n = 5000

# Simulated training dataset (batch export from data warehouse)
training_data = pd.DataFrame({
    'tenure_days': np.random.randint(1, 2000, n),
    'monthly_spend': np.random.uniform(20, 500, n),
    'support_tickets_30d': np.random.poisson(1.5, n),
    'plan_type': np.random.choice(['basic', 'pro', 'enterprise'], n, p=[0.5, 0.35, 0.15]),
    'days_since_last_login': np.random.exponential(7, n),
})

# Label: churned within 90 days
churn_prob = (
    0.3 * (training_data['days_since_last_login'] > 14).astype(float) +
    0.2 * (training_data['support_tickets_30d'] > 3).astype(float) +
    0.1 * (training_data['monthly_spend'] < 50).astype(float)
)
training_data['churned'] = (np.random.uniform(0, 1, n) < churn_prob).astype(int)

print(f'Dataset shape: {training_data.shape}')
print(f'Churn rate: {training_data["churned"].mean():.2%}')
training_data.head()

In [ ]:
# Training pipeline
X = training_data.drop('churned', axis=1)
y = training_data['churned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = ['tenure_days', 'monthly_spend', 'support_tickets_30d', 'days_since_last_login']
categorical_features = ['plan_type']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features),
])

model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(n_estimators=100, random_state=42))
])

model_pipeline.fit(X_train, y_train)

test_auc = roc_auc_score(y_test, model_pipeline.predict_proba(X_test)[:, 1])
print(f'Test AUC: {test_auc:.4f}')

joblib.dump(model_pipeline, 'churn_pipeline.joblib')
print('Pipeline saved.')

## Review questions

The pipeline above is **correctly implemented** — there is no data leakage and the full pipeline is saved.

However, in production, this model will receive features from a **live event stream API** rather than from the same batch export used for training.

Answer the following questions based on your review of the pipeline and the production scenario.

### Question 1: Skew entry points

For each feature below, describe one realistic production scenario where the feature value at inference time could differ from the feature value at training time, and explain whether the pipeline would detect the difference.

| Feature | Skew scenario | Would the pipeline detect it? |
|---------|--------------|-------------------------------|
| `support_tickets_30d` | | |
| `days_since_last_login` | | |
| `plan_type` | | |

### Question 2: The OrdinalEncoder unknown value

The pipeline uses `OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)` for `plan_type`.

If the company launches a new plan type called `'starter'` after this model is deployed, what value will the pipeline encode it as? What will the GradientBoostingClassifier do with this value? Is this the correct behaviour for your use case?

Write your answer below:

In [ ]:
# Demonstrate what happens with an unknown plan type
loaded_pipeline = joblib.load('churn_pipeline.joblib')

new_customer = pd.DataFrame([{
    'tenure_days': 45,
    'monthly_spend': 89.99,
    'support_tickets_30d': 0,
    'plan_type': 'starter',  # new plan type not seen at training time
    'days_since_last_login': 2.5
}])

# What does the pipeline predict?
prob = loaded_pipeline.predict_proba(new_customer)[0, 1]
print(f'Churn probability for unknown plan type: {prob:.4f}')

# What encoding does it receive?
enc = loaded_pipeline.named_steps['preprocessor'].named_transformers_['cat']
encoded = enc.transform(new_customer[['plan_type']])
print(f'Encoded plan_type value: {encoded[0, 0]}')

### Question 3: Detection strategy

Propose a monitoring strategy for this pipeline. For each type of skew you identified in Question 1, specify:
- The monitoring signal you would track
- The reference baseline (what does 'normal' look like?)
- The alert threshold you would set
- What action you would take when the alert fires

Write your strategy below:

### Question 4: Making it skew-resistant

The current pipeline saves the model artefact but does not save the training data statistics or feature distribution baseline.

Modify the training code below to also save:
1. The expected feature names and their training-time statistics (mean, std for numeric; value_counts for categorical)
2. A function that validates an inference request against these statistics before prediction

Hint: the `ColumnTransformer` contains the fitted scalers; you can extract their parameters.

In [ ]:
import json

# Your implementation here:
# 1. Extract training statistics from the fitted pipeline
# 2. Save them to a JSON contract file
# 3. Write a validate_request(X, contract) function that warns when a feature
#    is outside expected bounds

scaler = model_pipeline.named_steps['preprocessor'].named_transformers_['num']

# Extract the scaler statistics
training_stats = {
    'numeric_features': {
        feat: {'mean': float(mean), 'std': float(std)}
        for feat, mean, std in zip(numeric_features, scaler.mean_, scaler.scale_)
    },
    'categorical_features': {
        'plan_type': list(training_data['plan_type'].value_counts().index)
    }
}

print(json.dumps(training_stats, indent=2))

# TODO: write validate_request(X_inference, training_stats) that:
# - Checks all expected features are present
# - Warns if any numeric feature is more than 3 std deviations from training mean
# - Warns if any categorical feature has an unknown value